# Второй этап: BM25 в Elasticsearch

Индексация 5 183 целых документов SciFact, один BM25-запрос, пакетный поиск по test split и проверка тестами.

Вся основная логика живёт в `src/rag_playground`; ноутбук вызывает те же CLI-команды, которые можно запустить из терминала.

## 1. Подготовка окружения

Находим корень проекта, переходим в него и готовим вспомогательную функцию для вызова CLI.

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

# Ищем корень по `pyproject.toml`, чтобы ноутбук работал
# и при запуске из корня, и при запуске из папки notebooks.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / 'pyproject.toml').exists()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Не найден корень проекта с pyproject.toml')

os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

def run_command(*arguments: str, env: dict[str, str] | None = None) -> str:
    """Запустить модуль CLI тем же Python, которым запущен ноутбук."""
    completed = subprocess.run(
        [sys.executable, '-m', 'rag_playground.cli', *arguments],
        cwd=PROJECT_ROOT,
        env=env,
        check=True,
        capture_output=True,
        text=True,
    )
    return completed.stdout.strip()

print(f'Корень проекта: {PROJECT_ROOT}')
print(f'Python: {sys.version.split()[0]}')

Корень проекта: /Users/e.ilyina/Desktop/Портфолио/RAG
Python: 3.13.0


## 2. Параметры BM25

Показываем безопасную часть конфигурации: URL, имя индекса, batch size и top-K.

In [2]:
from rag_playground.config import Settings

# Явно читаем .env из корня, не выводя секреты GigaChat.
settings = Settings.from_env(PROJECT_ROOT / '.env')
safe_bm25_settings = {
    'elasticsearch_url': settings.elasticsearch_url,
    'index_name': settings.elasticsearch_index,
    'request_timeout_seconds': settings.elasticsearch_request_timeout,
    'index_batch_size': settings.bm25_batch_size,
    'manual_search_top_k': settings.bm25_default_top_k,
    'batch_run_top_k': settings.bm25_run_top_k,
}
print(json.dumps(safe_bm25_settings, ensure_ascii=False, indent=2))

{
  "elasticsearch_url": "http://localhost:9200",
  "index_name": "scifact-documents-v1",
  "request_timeout_seconds": 30,
  "index_batch_size": 500,
  "manual_search_top_k": 10,
  "batch_run_top_k": 100
}


## 3. Создание и наполнение индекса

`bm25-index` без `--recreate` безопасен для повторного запуска: он не удаляет существующий индекс, а обновляет документы по стабильному `_id = doc_id`.

In [3]:
# Загружаем весь корпус. Повторный запуск не создаёт дубликаты.
print(run_command('bm25-index'))

Index 'scifact-documents-v1' already exists; documents will be updated without deleting the index.
{
  "index_name": "scifact-documents-v1",
  "documents_read": 5183,
  "documents_indexed": 5183,
  "documents_failed": 0,
  "duration_seconds": 5.633529,
  "final_document_count": 5183
}


## 4. Проверка mapping и числа документов

В индексе должно быть 5 183 документа. Поиск идёт по `content` с `english` analyzer; исходные `title` и `text` хранятся для показа, но не индексируются отдельно.

In [4]:
# CLI возвращает версию Elasticsearch, count и точный mapping.
index_info = json.loads(run_command('bm25-info'))
print(json.dumps(index_info, ensure_ascii=False, indent=2))
assert index_info['index_exists'] is True
assert index_info['document_count'] == 5183
assert index_info['mapping']['properties']['content']['analyzer'] == 'english'

{
  "elasticsearch_version": "9.5.6",
  "index_name": "scifact-documents-v1",
  "index_exists": true,
  "document_count": 5183,
  "mapping": {
    "dynamic": "strict",
    "properties": {
      "content": {
        "type": "text",
        "analyzer": "english"
      },
      "doc_id": {
        "type": "keyword"
      },
      "text": {
        "type": "text",
        "index": false
      },
      "title": {
        "type": "text",
        "index": false
      }
    }
  }
}


## 5. Одиночный BM25-поиск

Запрос передаётся в `match` по полю `content`. Для каждого hit видны BM25-score, ранг, ID и исходный текст.

In [5]:
# Запускаем один запрос и сокращаем длинный abstract только для экрана.
search_output = json.loads(run_command(
    'bm25-search',
    'Vitamin D supplementation improves bone mineral density.',
    '--top-k',
    '5',
))
print(f"Задержка: {search_output['latency_ms']} мс")
for hit in search_output['hits']:
    preview = hit['text'][:220].replace('\n', ' ') + '...'
    print(f"\n#{hit['rank']} | doc_id={hit['doc_id']} | score={hit['score']:.3f}")
    print(hit['title'])
    print(preview)

Задержка: 22.81 мс

#1 | doc_id=4462079 | score=27.546
Estimation of optimal serum concentrations of 25-hydroxyvitamin D for multiple health outcomes.
Recent evidence suggests that vitamin D intakes above current recommendations may be associated with better health outcomes. However, optimal serum concentrations of 25-hydroxyvitamin D [25(OH)D] have not been defined. T...

#2 | doc_id=9555784 | score=27.243
Vitamin d deficiency in postmenopausal women - biological correlates.
INTRODUCTION Low vitamin D (VD) is associated with secondary hyperparathyroidism and both contribute to deleterious consequences (reduced bone mineral density (BMD), risk of fractures and falls). OBJECTIVE To study the V...

#3 | doc_id=13312471 | score=24.785
Vitamin D insufficiency and the blunted PTH response in established osteoporosis: the role of magnesium deficiency
Vitamin D insufficiency is common, however within individuals, not all manifest the biochemical effects of PTH excess. This further extends to 

## 6. Пакетный прогон test split

Из qrels выбираются 300 уникальных test-запросов. Для каждого запрашиваем top-100 и сохраняем ranking, сводку по запросам и манифест.

In [6]:
# Повторный запуск перезаписывает только артефакты с тем же run name.
run_paths = json.loads(run_command('bm25-run', '--split', 'test', '--top-k', '100'))
print(json.dumps(run_paths, ensure_ascii=False, indent=2))

{
  "ranking": "reports/results/runs/bm25_test_top100.jsonl",
  "queries": "reports/results/runs/bm25_test_top100_queries.jsonl",
  "manifest": "reports/results/runs/bm25_test_top100_manifest.json"
}


## 7. Проверка артефактов

В ranking может быть меньше `300 × 100` строк: BM25 не обязан возвращать документы с нулевым score. При контрольном запуске один запрос имел 66 hits, а остальные — 100.

In [7]:
# Читаем файлы обратно и проверяем ключевые инварианты запуска.
ranking_path = PROJECT_ROOT / run_paths['ranking']
queries_path = PROJECT_ROOT / run_paths['queries']
manifest_path = PROJECT_ROOT / run_paths['manifest']
ranking_rows = [json.loads(line) for line in ranking_path.read_text(encoding='utf-8').splitlines()]
query_rows = [json.loads(line) for line in queries_path.read_text(encoding='utf-8').splitlines()]
run_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))

assert len(query_rows) == 300
assert len({row['query_id'] for row in query_rows}) == 300
assert len(ranking_rows) == run_manifest['ranking_rows']
assert all(1 <= row['rank'] <= 100 for row in ranking_rows)
print(json.dumps(run_manifest, ensure_ascii=False, indent=2))
print('\nПример первых трёх строк ranking:')
for row in ranking_rows[:3]:
    print(row)

{
  "analyzer": "english",
  "created_at_utc": "2026-11-21T10:03:02.816329+00:00",
  "elasticsearch_version": "9.5.6",
  "index_name": "scifact-documents-v1",
  "indexed_documents": 5183,
  "mean_query_latency_ms": 17.827,
  "queries": 300,
  "ranking_rows": 29966,
  "retrieval_method": "bm25",
  "split": "test",
  "top_k": 100
}

Пример первых трёх строк ranking:
{'doc_id': '10608397', 'query_id': '1', 'rank': 1, 'retrieval_method': 'bm25', 'score': 13.339636}
{'doc_id': '18953920', 'query_id': '1', 'rank': 2, 'retrieval_method': 'bm25', 'score': 10.022104}
{'doc_id': '34386619', 'query_id': '1', 'rank': 3, 'retrieval_method': 'bm25', 'score': 9.883365}


## 8. Автоматические проверки

Запускаем Ruff, все быстрые тесты и отдельный integration-тест. Последний создаёт в Elasticsearch временный индекс с точным уникальным именем и удаляет только его.

In [ ]:
# Общий helper для команд, которые не являются подкомандами CLI.
def run_tool(*arguments: str, env: dict[str, str] | None = None) -> str:
    completed = subprocess.run(
        list(arguments), cwd=PROJECT_ROOT, env=env, check=True,
        capture_output=True, text=True,
    )
    return (completed.stdout + completed.stderr).strip()

print('Ruff:')
print(run_tool(sys.executable, '-m', 'ruff', 'check', 'src', 'tests'))
print('\nPytest:')
print(run_tool(sys.executable, '-m', 'pytest'))
print('\nIntegration test:')
integration_env = os.environ.copy()
integration_env['RUN_ELASTICSEARCH_INTEGRATION'] = '1'
print(run_tool(
    sys.executable, '-m', 'pytest', 'tests/integration/test_elasticsearch_bm25.py',
    env=integration_env,
))

Ruff:


All checks passed!

Pytest:


s....................                                                    [100%]
20 passed, 1 skipped in 0.61s

Integration test:


.                                                                        [100%]
1 passed in 1.82s


## 9. Итог

Второй этап завершён: корпус проиндексирован на уровне целых документов, BM25-поиск работает для одиночного и batch-режима, а каждый пакетный запуск оставляет воспроизводимые артефакты.

Recall@K, nDCG@K и MRR здесь пока не считаются: общий evaluation-модуль будет добавлен на третьем этапе, чтобы BM25, dense и hybrid оценивались одинаково.